# 🤖 LLM News Scoring & Translation

**Model:** Typhoon2.5-Qwen3-4B (Thai-capable)
**Tasks:** Score articles + Translate to Thai

## Pipeline
```
Scraped CSV → LLM Scoring → Prioritized Articles → Thai Translation
```

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL 1: SETUP
# ════════════════════════════════════════════════════════════════════════════════
# Install llama-cpp-python if needed (with GPU support if available)
# !pip install llama-cpp-python

import json
import os
import pandas as pd
from datetime import datetime, timedelta
from tqdm.notebook import tqdm
from llama_cpp import Llama

print("✅ Imports Ready")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL 2: CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════
class Config:
    # Paths
    MODEL_PATH = "../data/models/typhoon2.5-qwen3-4b-q4_k_m.gguf"
    INPUT_CSV = "../v10_Scraper_Hybrid/scraped_news_v10.csv"
    OUTPUT_CSV = "scored_articles.csv"
    
    # Model Settings
    N_CTX = 4096          # Context window
    N_GPU_LAYERS = 0      # Set to -1 for full GPU, 0 for CPU only
    TEMPERATURE = 0.1     # Low = more deterministic
    MAX_TOKENS = 1024
    
    # Scoring Thresholds
    FRESHNESS_DAYS = 7    # Articles < 7 days = fresh
    MIN_SCORE_FOR_TRANSLATION = 4  # Only translate if score >= 4

CFG = Config()
print(f"📁 Model: {CFG.MODEL_PATH}")
print(f"📄 Input: {CFG.INPUT_CSV}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL 3: LOAD MODEL
# ════════════════════════════════════════════════════════════════════════════════
print("🔄 Loading Typhoon2.5 model (this may take a minute)...")

llm = Llama(
    model_path=CFG.MODEL_PATH,
    n_ctx=CFG.N_CTX,
    n_gpu_layers=CFG.N_GPU_LAYERS,
    verbose=False
)

print("✅ Model Loaded!")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL 4: SCORING PROMPT
# ════════════════════════════════════════════════════════════════════════════════
SCORING_PROMPT = '''You are a professional news reviewer and AI/Business analyst.

Analyze this article and return ONLY valid JSON (no explanation):

---
URL: {url}
Title: {title}
Author: {author}
Published: {published}
Content:
{content}
---

Current Date: {current_date}

Return JSON:
{{
  "Is_fresh": 0 or 1 (1 if published within 7 days),
  "Relevance": 0 or 1 (1 if relevant to AI/Tech/Business),
  "AI_News": 0 or 1 (1 if about AI/ML/LLM),
  "Social_Impact": 0 or 1 (1 if affects society),
  "Economic_Impact": 0 or 1 (1 if affects economy/markets),
  "No_ads": 0 or 1 (1 if content is clean, not promotional),
  "Is_reference": 0 or 1 (1 if cites sources/data),
  "Keywords": ["keyword1", "keyword2", "keyword3"],
  "Lead": "Who/What/Where/When/Why summary in <=40 words"
}}'''

print("✅ Scoring Prompt Ready")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL 5: TRANSLATION PROMPT
# ════════════════════════════════════════════════════════════════════════════════
TRANSLATION_PROMPT = '''You are a professional Thai translator.

Translate this news article to Thai. Follow these rules:
1. Names: Show both Thai and English, e.g., "สตีฟ จ็อบส์ (Steve Jobs)"
2. Abbreviations: Explain on first use, e.g., "ประธานาธิบดีสหรัฐ (POTUS)"
3. Keep technical terms accurate
4. Maintain journalistic style

---
Title: {title}
Content:
{content}
---

Return JSON:
{{
  "title_th": "Thai title",
  "content_th": "Thai translation of content",
  "source_line": "ที่มา: {source} — {url} (เผยแพร่ {published})"
}}'''

print("✅ Translation Prompt Ready")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL 6: LLM FUNCTIONS
# ════════════════════════════════════════════════════════════════════════════════
def call_llm(prompt: str) -> str:
    """Call the LLM and return response."""
    try:
        output = llm(
            prompt,
            max_tokens=CFG.MAX_TOKENS,
            temperature=CFG.TEMPERATURE,
            stop=["```", "</s>"],
            echo=False
        )
        return output['choices'][0]['text'].strip()
    except Exception as e:
        print(f"❌ LLM Error: {e}")
        return ""

def parse_json_response(text: str) -> dict:
    """Extract JSON from LLM response."""
    try:
        # Find JSON block
        start = text.find('{')
        end = text.rfind('}') + 1
        if start != -1 and end > start:
            json_str = text[start:end]
            return json.loads(json_str)
    except:
        pass
    return {}

def score_article(article: dict) -> dict:
    """Score a single article."""
    prompt = SCORING_PROMPT.format(
        url=article.get('url', ''),
        title=article.get('headline', ''),
        author=article.get('author', 'Unknown'),
        published=article.get('published', 'Unknown'),
        content=article.get('full_content', '')[:2000],  # Limit content
        current_date=datetime.now().strftime('%Y-%m-%d')
    )
    
    response = call_llm(prompt)
    result = parse_json_response(response)
    
    # Calculate total score
    score_fields = ['Is_fresh', 'Relevance', 'AI_News', 'Social_Impact', 
                    'Economic_Impact', 'No_ads', 'Is_reference']
    total_score = sum(result.get(f, 0) for f in score_fields)
    result['total_score'] = total_score
    
    return result

def translate_article(article: dict) -> dict:
    """Translate article to Thai."""
    prompt = TRANSLATION_PROMPT.format(
        title=article.get('headline', ''),
        content=article.get('full_content', '')[:2000],
        source=article.get('source', ''),
        url=article.get('url', ''),
        published=article.get('published', '')
    )
    
    response = call_llm(prompt)
    return parse_json_response(response)

print("✅ LLM Functions Ready")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL 7: MAIN PIPELINE
# ════════════════════════════════════════════════════════════════════════════════
def run_scoring_pipeline():
    """Score all articles from scraped CSV."""
    # Load data
    if not os.path.exists(CFG.INPUT_CSV):
        print(f"❌ Input file not found: {CFG.INPUT_CSV}")
        return None
    
    df = pd.read_csv(CFG.INPUT_CSV)
    print(f"📊 Loaded {len(df)} articles")
    
    # Score each article
    results = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Scoring"):
        article = row.to_dict()
        score = score_article(article)
        
        # Merge score with original data
        result = {
            **article,
            'Is_fresh': score.get('Is_fresh', 0),
            'Relevance': score.get('Relevance', 0),
            'AI_News': score.get('AI_News', 0),
            'Social_Impact': score.get('Social_Impact', 0),
            'Economic_Impact': score.get('Economic_Impact', 0),
            'No_ads': score.get('No_ads', 0),
            'Is_reference': score.get('Is_reference', 0),
            'total_score': score.get('total_score', 0),
            'keywords_llm': ', '.join(score.get('Keywords', [])),
            'lead': score.get('Lead', '')
        }
        results.append(result)
    
    # Save results
    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values('total_score', ascending=False)
    result_df.to_csv(CFG.OUTPUT_CSV, index=False)
    
    print(f"\n💾 Saved to {CFG.OUTPUT_CSV}")
    print(f"\n📊 Score Distribution:")
    print(result_df['total_score'].value_counts().sort_index())
    
    return result_df

# Run it
scored_df = run_scoring_pipeline()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL 8: TRANSLATION PIPELINE
# ════════════════════════════════════════════════════════════════════════════════
def run_translation_pipeline(df: pd.DataFrame, min_score: int = 4):
    """Translate top-scoring articles."""
    # Filter high-scoring articles
    top_articles = df[df['total_score'] >= min_score].head(10)
    print(f"🎯 Translating {len(top_articles)} articles (score >= {min_score})")
    
    translations = []
    for idx, row in tqdm(top_articles.iterrows(), total=len(top_articles), desc="Translating"):
        article = row.to_dict()
        translation = translate_article(article)
        
        translations.append({
            'url': article.get('url'),
            'headline_en': article.get('headline'),
            'headline_th': translation.get('title_th', ''),
            'content_th': translation.get('content_th', ''),
            'source_line': translation.get('source_line', ''),
            'total_score': article.get('total_score')
        })
    
    # Save
    trans_df = pd.DataFrame(translations)
    trans_df.to_csv('translated_articles.csv', index=False)
    print(f"💾 Saved translations to translated_articles.csv")
    
    return trans_df

# Run if scoring is done
if scored_df is not None and len(scored_df) > 0:
    trans_df = run_translation_pipeline(scored_df)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL 9: VIEW RESULTS
# ════════════════════════════════════════════════════════════════════════════════
def show_top_articles():
    if not os.path.exists(CFG.OUTPUT_CSV):
        print("No scored articles yet.")
        return
    
    df = pd.read_csv(CFG.OUTPUT_CSV)
    top = df.nlargest(5, 'total_score')
    
    print("═" * 60)
    print("🏆 TOP SCORED ARTICLES")
    print("═" * 60)
    
    for i, (_, row) in enumerate(top.iterrows(), 1):
        print(f"\n{i}. [{row['total_score']}/7] {row['headline'][:50]}...")
        print(f"   📰 {row['source']}")
        print(f"   📝 {row.get('lead', '')[:80]}...")

show_top_articles()